In [120]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_openai.chat_models import ChatOpenAI
from langchain_chroma import Chroma 
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os 

load_dotenv()

api_key =  os.environ["OPENAI_API_KEY"]
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
    )

llm = ChatOpenAI(model="gpt-4.1")

In [121]:
pages =  PyMuPDFLoader("Tithis.pdf").load()
doc =  "\n".join(p.page_content for p in pages)



sematic_chunker  =  SemanticChunker(
    embeddings,breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=1.5
)


docs  =  sematic_chunker.create_documents([doc])

print(docs)

[Document(metadata={}, page_content="PA N C H A N G S E C R E T S · J A N M A T I T H I\nJanma Tithi Se Jaane\nAapka Asli Swabhav\nजन्म तिथि और आपका स्वभाव\nThe 16 lunar days of birth — ruling deity, core personality, quirks and remedies\nThe Significance of Janma Tithi: While most people know their English calendar birthdate, Vedic\nastrology places paramount importance on the Janma Tithi (Lunar Day at the time of birth), determined by\nthe longitudinal distance between the Sun and Moon. The 16 Tithis: Across the two lunar fortnights (Shukla Paksha and Krishna Paksha), there are 16 distinct\nTithis—from Pratipada to Chaturdashi, ending in Purnima (Full Moon) and Amavasya (New Moon). Core Philosophy: Every Tithi carries an inherent energetic vibration and is governed by a specific ruling\ndeity (Ishta Devata) that defines psychological patterns, behavioral strengths/weaknesses, and karmic\nremedies. T H E 1 6 T I T H I S\n1\nPratipada प्रतिपदा\n2\nDwitiya द्वितीया\n3\nTritiya तृतीया\n4

17


In [ ]:


vectorstore = Chroma.from_documents(documents=docs,embedding=embeddings,persist_directory="./data/tithis")

multiquery =  MultiQueryRetriever.from_llm(
     retriever=vectorstore.as_retriever(search_kwargs={"k":2}),
    llm=llm,
    include_original=True
)

prompt_template =  ChatPromptTemplate.from_template(""""
       Answer using the context :  {context} and query : {query}                                             
""")





query =  " in which tithi bold and courageous person is born ? "
context  =  multiquery.invoke(query)[0].page_content



chain =  prompt_template |  llm |  StrOutputParser()
result = chain.invoke({"context":context,"query":query})

print(result)


According to the provided context, a **bold and courageous person** is most likely to be **born on Pratipada Tithi (प्रतिपदा तिथि)**, the 1st Lunar Day.

**Supporting Details from the Text:**
- Pratipada Tithi people are described as:
    - "Natural initiators and trailblazers"
    - "Fiercely independent and quick to take bold risks (e.g., fighting family for inter-caste love)."
    - "Take unconventional career or life paths that no one in their family or ancestry has ever attempted."

These traits indicate boldness and courage.

**Summary:**  
**Pratipada Tithi** is the tithi in which bold and courageous persons are born.


In [132]:
print(type(vectorstore))

<class 'langchain_chroma.vectorstores.Chroma'>
